In [16]:
import os  # 导入os模块，用于处理文件路径
import pandas as pd  # 导入pandas库，用于数据处理
import numpy as np  # 导入numpy库，用于数值计算
from sklearn.model_selection import train_test_split  # 导入train_test_split函数，用于数据集划分
from sklearn.ensemble import RandomForestRegressor  # 导入随机森林回归模型
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error  # 导入均方误差评估指标
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
import pickle  # 导入pickle模块，用于模型保存和加载
from sklearn.model_selection import KFold


# 统一输出路径
output_dir = r"E:\论文\综合长势\超参数优化\SSA"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

class SSA:
    def __init__(self, func, bounds, population_size=14, iterations=10,random_seed=36):
        self.func = func  # 传入的评估函数
        self.bounds = bounds  # 参数搜索范围
        self.population_size = population_size  # 种群大小
        self.iterations = iterations  # 迭代次数
        self.dim = len(bounds)  # 参数维度
        self.best_score = float('inf')  # 初始最佳得分设为无穷大000
        self.best_pos = None  # 初始最佳位置为空
        self.output_dir = output_dir  # 使用全局路径
        np.random.seed(random_seed)  # 设置全局随机数种子
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
        self.best_params_per_iteration = []  # 存储每一代的最佳参数

    def initialize(self):
        population = np.random.rand(self.population_size, self.dim)  # 随机初始化种群
        for i in range(self.dim):
            population[:, i] = population[:, i] * (self.bounds[i][1] - self.bounds[i][0]) + self.bounds[i][0]  # 将种群参数初始化到指定范围内
        return population

    def evaluate(self, population):
        fitness = np.apply_along_axis(self.func, 1, population)  # 对种群中的个体进行评估
        return fitness

    def update(self, population, fitness):
        best_index = np.argmin(fitness)  # 找到种群中最佳个体的索引
        best_score = fitness[best_index]  # 最佳个体的评估得分
        best_pos = population[best_index].copy()  # 最佳个体的位置
        return best_score, best_pos


    def run(self):
        population = self.initialize()  # 初始化种群
        fitness = self.evaluate(population)  # 评估种群
        self.best_score, self.best_pos = self.update(population, fitness)  # 更新最佳个体信息
        avg_oob_errors = []  # 用于存储每次迭代的平均 OOB Error
        for t in range(self.iterations):


            # 用于存储每代中的麻雀位置和MSE值
            iteration_data = []

            r1 = np.random.rand(self.population_size, self.dim)
            r2 = np.random.rand(self.population_size, self.dim)
            p = np.random.rand(self.population_size, self.dim)
            q = np.random.rand(self.population_size, self.dim)

            for i in range(self.population_size):
                if p[i][0] < 0.8:
                    if r2[i][0] < 0.5:
                        population[i] = self.best_pos + np.abs(population[i] - self.best_pos) * np.exp(t / self.iterations - 1)
                    else:
                        population[i] = population[i] + np.abs(population[i] - self.best_pos) * np.exp(t / self.iterations - 1)
                else:
                    if q[i][0] < 0.5:
                        population[i] = population[i] + np.random.randn() * np.abs(population[i] - self.best_pos)
                    else:
                        population[i] = self.best_pos + np.random.randn() * np.abs(population[i] - self.best_pos)

                for d in range(self.dim):
                    if population[i, d] < self.bounds[d][0]:
                        population[i, d] = self.bounds[d][0]
                    if population[i, d] > self.bounds[d][1]:
                        population[i, d] = self.bounds[d][1]

            fitness = self.evaluate(population)  # 更新种群评估
            # 获取当前代最佳个体信息并记录
            current_best_score, current_best_pos = self.update(population, fitness)
            self.best_params_per_iteration.append({
                'iteration': t + 1,
                'best_n_estimators': int(current_best_pos[0]),
                'best_max_depth': int(current_best_pos[1]),
                'best_min_samples_split': int(current_best_pos[2]),
                'best_min_samples_leaf': int(current_best_pos[3]),
                'best_OOB': current_best_score
            })
            # 计算平均 OOB Error 并记录
            avg_oob_error = np.mean(fitness)
            avg_oob_errors.append({'iteration': t + 1, 'avg_oob_error': avg_oob_error})
            # 记录当前种群中每个个体的参数值和对应的MSE值
            for i in range(self.population_size):
                iteration_data.append({
                    'n_estimators': int(population[i][0]),
                    'max_depth': int(population[i][1]),
                    'min_samples_split': int(population[i][2]),
                    'min_samples_leaf': int(population[i][3]),
                    'OOB': fitness[i]
                })
                # 保存当前代的所有麻雀及其对应的 OOB 误差值为 Excel 文件
                df = pd.DataFrame(iteration_data)
                file_path = os.path.join(self.output_dir, f'iteration_{t + 1}.xlsx')
                df.to_excel(file_path, index=False)


            if current_best_score < self.best_score:
                self.best_score = current_best_score
                self.best_pos = current_best_pos

            best_params_df = pd.DataFrame(self.best_params_per_iteration)
            best_params_file_path = os.path.join(self.output_dir, 'best_params_per_iteration.xlsx')
            best_params_df.to_excel(best_params_file_path, index=False)

            # 保存平均 OOB Error 到 Excel 文件
            avg_oob_errors_df = pd.DataFrame(avg_oob_errors)
            avg_oob_errors_file_path = os.path.join(self.output_dir, 'avg_oob_errors_per_iteration.xlsx')
            avg_oob_errors_df.to_excel(avg_oob_errors_file_path, index=False)

        return self.best_pos, self.best_score


In [35]:
# 读取数据
data = pd.read_csv('E:\论文\综合长势\超参数优化\Total.csv',header=None)

# 将DataFrame转换为numpy数组
data_array = data.to_numpy()  # 使用滤波后的数据
X = data_array[:, 0:-1]  # 提取特征X，去除最后一列作为特征数据
Y = data_array[:, -1]  # 提取标签Y，最后一列作为目标变量
z=36
# 重采样数据
X_resampled, Y_resampled = resample(X, Y, replace=True, n_samples=len(X) * 2, random_state=z)

# 划分重取样后的数据集
scaler = MinMaxScaler()  # 初始化 MinMaxScaler
X_std_resampled = scaler.fit_transform(X_resampled)  # 归一化
x_train, x_test, y_train, y_test = train_test_split(X_std_resampled, Y_resampled, test_size=0.2, random_state=z)#380

# 保存 MinMaxScaler 的归一化模型参数
scaler_min = scaler.data_min_
scaler_scale = scaler.data_range_


def evaluate(params):
    n_estimators, max_depth, min_samples_split, min_samples_leaf = (
        int(params[0]), int(params[1]), int(params[2]), int(params[3])
    )

    # 启用袋外评分
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),  # 确保是整数
        min_samples_leaf=int(min_samples_leaf),  # 确保是整数
        random_state=z,
        oob_score=True
    )

    rf.fit(x_train, y_train)  # 使用增样后的训练集

    # 使用袋外评分进行评估
    oob_error = 1 - rf.oob_score_  # 1 - oob_score表示袋外误差

    return oob_error

# 修改 SSA 中的搜索范围
bounds = [(1, 200),
     (1, 10),
     (2, 10),
     (1, 10)]

# 初始化SSA
ssa = SSA(evaluate, bounds, population_size=20
                               , iterations=9)#12-8 12-12 8-12 8-8           12-10/12  8-8 

# 运行 SSA 优化
best_params, best_score = ssa.run()
best_n_estimators, best_max_depth = int(best_params[0]), int(best_params[1])
best_min_samples_split, best_min_samples_leaf = int(best_params[2]), int(best_params[3])

print(f'Best n_estimators: {best_n_estimators}')
print(f'Best max_depth: {best_max_depth}')
print(f'Best min_samples_split: {best_min_samples_split}')
print(f'Best min_samples_leaf: {best_min_samples_leaf}')
print(f'Best OOB error: {best_score}')


Best n_estimators: 195
Best max_depth: 10
Best min_samples_split: 6
Best min_samples_leaf: 1
Best OOB error: 0.12948922534806395


In [3]:
import os  # 导入os模块，用于处理文件路径
import pandas as pd  # 导入pandas库，用于数据处理
import numpy as np  # 导入numpy库，用于数值计算
from sklearn.model_selection import train_test_split  # 导入train_test_split函数，用于数据集划分
from sklearn.ensemble import RandomForestRegressor  # 导入随机森林回归模型
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error  # 导入均方误差评估指标
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
import pickle  # 导入pickle模块，用于模型保存和加载
from sklearn.model_selection import KFold


# 统一输出路径
output_dir = r"E:\论文\综合长势\数据\SSA-RF"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

class SSA:
    def __init__(self, func, bounds, population_size=30, iterations=50,random_seed=42):
        self.func = func  # 传入的评估函数
        self.bounds = bounds  # 参数搜索范围
        self.population_size = population_size  # 种群大小
        self.iterations = iterations  # 迭代次数
        self.dim = len(bounds)  # 参数维度
        self.best_score = float('inf')  # 初始最佳得分设为无穷大000
        self.best_pos = None  # 初始最佳位置为空
        self.output_dir = output_dir  # 使用全局路径
        np.random.seed(random_seed)  # 设置全局随机数种子
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
        self.best_params_per_iteration = []  # 存储每一代的最佳参数

    def initialize(self):
        population = np.random.rand(self.population_size, self.dim)  # 随机初始化种群
        for i in range(self.dim):
            population[:, i] = population[:, i] * (self.bounds[i][1] - self.bounds[i][0]) + self.bounds[i][0]  # 将种群参数初始化到指定范围内
        return population

    def evaluate(self, population):
        fitness = np.apply_along_axis(self.func, 1, population)  # 对种群中的个体进行评估
        return fitness

    def update(self, population, fitness):
        best_index = np.argmin(fitness)  # 找到种群中最佳个体的索引
        best_score = fitness[best_index]  # 最佳个体的评估得分
        best_pos = population[best_index].copy()  # 最佳个体的位置
        return best_score, best_pos


    def run(self):
        population = self.initialize()  # 初始化种群
        fitness = self.evaluate(population)  # 评估种群
        self.best_score, self.best_pos = self.update(population, fitness)  # 更新最佳个体信息
        avg_oob_errors = []  # 用于存储每次迭代的平均 OOB Error
        for t in range(self.iterations):


            # 用于存储每代中的麻雀位置和MSE值
            iteration_data = []

            r1 = np.random.rand(self.population_size, self.dim)
            r2 = np.random.rand(self.population_size, self.dim)
            p = np.random.rand(self.population_size, self.dim)
            q = np.random.rand(self.population_size, self.dim)

            for i in range(self.population_size):
                if p[i][0] < 0.8:
                    if r2[i][0] < 0.5:
                        population[i] = self.best_pos + np.abs(population[i] - self.best_pos) * np.exp(t / self.iterations - 1)
                    else:
                        population[i] = population[i] + np.abs(population[i] - self.best_pos) * np.exp(t / self.iterations - 1)
                else:
                    if q[i][0] < 0.5:
                        population[i] = population[i] + np.random.randn() * np.abs(population[i] - self.best_pos)
                    else:
                        population[i] = self.best_pos + np.random.randn() * np.abs(population[i] - self.best_pos)

                for d in range(self.dim):
                    if population[i, d] < self.bounds[d][0]:
                        population[i, d] = self.bounds[d][0]
                    if population[i, d] > self.bounds[d][1]:
                        population[i, d] = self.bounds[d][1]

            fitness = self.evaluate(population)  # 更新种群评估
            # 获取当前代最佳个体信息并记录
            current_best_score, current_best_pos = self.update(population, fitness)
            self.best_params_per_iteration.append({
                'iteration': t + 1,
                'best_n_estimators': int(current_best_pos[0]),
                'best_max_depth': int(current_best_pos[1]),
                'best_min_samples_split': int(current_best_pos[2]),
                'best_min_samples_leaf': int(current_best_pos[3]),
                'best_OOB': current_best_score
            })
            # 计算平均 OOB Error 并记录
            avg_oob_error = np.mean(fitness)
            avg_oob_errors.append({'iteration': t + 1, 'avg_oob_error': avg_oob_error})
            # 记录当前种群中每个个体的参数值和对应的MSE值
            for i in range(self.population_size):
                iteration_data.append({
                    'n_estimators': int(population[i][0]),
                    'max_depth': int(population[i][1]),
                    'min_samples_split': int(population[i][2]),
                    'min_samples_leaf': int(population[i][3]),
                    'OOB': fitness[i]
                })
                # 保存当前代的所有麻雀及其对应的 OOB 误差值为 Excel 文件
                df = pd.DataFrame(iteration_data)
                file_path = os.path.join(self.output_dir, f'iteration_{t + 1}.xlsx')
                df.to_excel(file_path, index=False)


            if current_best_score < self.best_score:
                self.best_score = current_best_score
                self.best_pos = current_best_pos

            best_params_df = pd.DataFrame(self.best_params_per_iteration)
            best_params_file_path = os.path.join(self.output_dir, 'best_params_per_iteration.xlsx')
            best_params_df.to_excel(best_params_file_path, index=False)

            # 保存平均 OOB Error 到 Excel 文件
            avg_oob_errors_df = pd.DataFrame(avg_oob_errors)
            avg_oob_errors_file_path = os.path.join(self.output_dir, 'avg_oob_errors_per_iteration.xlsx')
            avg_oob_errors_df.to_excel(avg_oob_errors_file_path, index=False)

        return self.best_pos, self.best_score

# 读取数据
data = pd.read_csv('E:\论文\综合长势\数据\SSA-RF\全部数据相关性分析.csv',header=None)

# 将DataFrame转换为numpy数组
data_array = data.to_numpy()  # 使用滤波后的数据
X = data_array[:, 0:-1]  # 提取特征X，去除最后一列作为特征数据
Y = data_array[:, -1]  # 提取标签Y，最后一列作为目标变量

# 重采样数据
X_resampled, Y_resampled = resample(X, Y, replace=True, n_samples=len(X) * 2, random_state=42)

# 划分重取样后的数据集
scaler = MinMaxScaler()  # 初始化 MinMaxScaler
X_std_resampled = scaler.fit_transform(X_resampled)  # 归一化
x_train, x_test, y_train, y_test = train_test_split(X_std_resampled, Y_resampled, test_size=0.2, random_state=42)

# 保存 MinMaxScaler 的归一化模型参数
scaler_min = scaler.data_min_
scaler_scale = scaler.data_range_


def evaluate(params):
    n_estimators, max_depth, min_samples_split, min_samples_leaf = (
        int(params[0]), int(params[1]), int(params[2]), int(params[3])
    )

    # 启用袋外评分
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),  # 确保是整数
        min_samples_leaf=int(min_samples_leaf),  # 确保是整数
        random_state=42,
        oob_score=True
    )

    rf.fit(x_train, y_train)  # 使用增样后的训练集

    # 使用袋外评分进行评估
    oob_error = 1 - rf.oob_score_  # 1 - oob_score表示袋外误差

    return oob_error

# 修改 SSA 中的搜索范围
bounds = [
(20, 1000),
     (2, 100),
     (2, 50),
     (1, 50)]

# 初始化SSA
ssa = SSA(evaluate, bounds, population_size=30, iterations=50)

# 运行 SSA 优化
best_params, best_score = ssa.run()
best_n_estimators, best_max_depth = int(best_params[0]), int(best_params[1])
best_min_samples_split, best_min_samples_leaf = int(best_params[2]), int(best_params[3])

print(f'Best n_estimators: {best_n_estimators}')
print(f'Best max_depth: {best_max_depth}')
print(f'Best min_samples_split: {best_min_samples_split}')
print(f'Best min_samples_leaf: {best_min_samples_leaf}')
print(f'Best OOB error: {best_score}')

# 使用最优参数训练模型
rf_best = RandomForestRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_split=best_min_samples_split,
    min_samples_leaf=best_min_samples_leaf,
    random_state=42
)
rf_best.fit(x_train, y_train)


# 使用测试集进行预测并计算 R^2
y_pred_test = rf_best.predict(x_test)
r2_best = r2_score(y_test, y_pred_test)
mae_best = mean_absolute_error(y_test, y_pred_test)
mse_best = mean_squared_error(y_test, y_pred_test)  # 计算 MSE

print(f"模型的 R^2: {r2_best}")
print(f"模型的 MAE: {mae_best}")
print(f"模型的 MSE: {mse_best}")
# 保存 R^2 和预测结果到 Excel 文件
results_df = pd.DataFrame({
    'True Value': y_test,
    'Predicted Value': y_pred_test
})
results_df['R^2'] = r2_best
results_df['MAE'] = mae_best
results_df['MSE'] = mse_best  # 添加 MSE

output_path_R_MAE_MSE = os.path.join(output_dir, 'predictions_vs_true_values_with_r2_MAE_MSE.xlsx')
results_df.to_excel(output_path_R_MAE_MSE, index=False)

print(f"测试集的预测值、R^2、MAE 和 MSE 已保存到 {output_path_R_MAE_MSE} 文件中")

# 使用训练集进行预测并计算 R^2、MAE 和 MSE
y_pred_train = rf_best.predict(x_train)  # 使用训练集进行预测
r2_train = r2_score(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)  # 计算 MSE

# 输出训练集的 R^2、MAE 和 MSE
print(f"训练集的 R^2: {r2_train}")
print(f"训练集的 MAE: {mae_train}")
print(f"训练集的 MSE: {mse_train}")

# 保存 R^2、MAE、MSE 和预测结果到 Excel 文件
results_df = pd.DataFrame({
    'True Value': y_train,
    'Predicted Value': y_pred_train
})
results_df['R^2'] = r2_train
results_df['MAE'] = mae_train
results_df['MSE'] = mse_train  # 添加 MSE

# 定义文件保存路径
output_path_R_MAE_MSE_train = os.path.join(output_dir, 'predictions_vs_true_values_train_with_r2_MAE_MSE.xlsx')
results_df.to_excel(output_path_R_MAE_MSE_train, index=False)

print(f"训练集的预测值、R^2、MAE 和 MSE 已保存到 {output_path_R_MAE_MSE_train} 文件中")


with open('best_random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_best, f)  # 将最优模型保存到文件中
print('模型已保存到 best_random_forest_model.pkl')  # 打印保存成功信息

# 获取特征的重要性
feature_importances = rf_best.feature_importances_

# 假设特征变量的名称为"Feature 1", "Feature 2", ... "Feature n"，可以根据您的特征实际名称进行调整
feature_names = ["底部含水层厚度", "底部含水层富水性", "底部含水层富水压", "底部黏土层厚度","基岩厚度","煤层倾角","煤层埋藏深度","松基比","松深比","开采高度","工作面长度","开采方法"]

# 创建DataFrame并保存到Excel
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# 保存特征重要性到Excel文件
output_path = os.path.join(output_dir, 'feature_importances.xlsx')
feature_importance_df.to_excel(output_path, index=False)

print(f"特征变量重要性已保存到 {output_path} 文件中")



# 重采样数据
X_resampled1, Y_resampled1 = resample(X, Y, replace=True, n_samples=len(X) * 2, random_state=42)

X_std_resampled1 = scaler.fit_transform(X_resampled1)  # 归一化

# 使用最佳参数组合进行十折交叉验证
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 初始化用于存储每折的预测结果和评估指标
cv_results = pd.DataFrame(columns=['Fold', 'True Value', 'Predicted Value'])
metrics = {'Fold': [], 'R²': [], 'MAE': [], 'MSE': []}

# 进行十折交叉验证
fold_number = 1
for train_index, val_index in kf.split(X_std_resampled1):
    X_train, X_val = X_std_resampled1[train_index], X_std_resampled1[val_index]
    y_train, y_val = Y_resampled1[train_index], Y_resampled1[val_index]

    # 使用最佳参数组合训练Lasso模型
    rf_best.fit(X_train, y_train)

    # 预测验证集
    y_pred_val = rf_best.predict(X_val)

    # 计算每折的 R²、MAE 和 MSE
    r2 = r2_score(y_val, y_pred_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    mse = mean_squared_error(y_val, y_pred_val)

    # 将每折的预测结果添加到cv_results DataFrame中
    fold_results = pd.DataFrame({
        'Fold': fold_number,
        'True Value': y_val,
        'Predicted Value': y_pred_val
    })
    cv_results = pd.concat([cv_results, fold_results], ignore_index=True)

    # 将每折的评估指标添加到指标字典中
    metrics['Fold'].append(fold_number)
    metrics['R²'].append(r2)
    metrics['MAE'].append(mae)
    metrics['MSE'].append(mse)

    fold_number += 1

# 保存预测结果和指标到 Excel 文件
output_path_cv_results = os.path.join(output_dir, 'SSA-RF_cross_validation_results.xlsx')
with pd.ExcelWriter(output_path_cv_results) as writer:
    # 保存每折的预测结果
    cv_results.to_excel(writer, sheet_name='Predictions', index=False)

    # 将每折的评估指标转换为 DataFrame，并保存到同一文件中的不同工作表
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_excel(writer, sheet_name='Metrics', index=False)

    # 计算 R²、MAE 和 MSE 的平均值，并保存
    avg_metrics = pd.DataFrame({
        'Metric': ['R²', 'MAE', 'MSE'],
        'Average': [metrics_df['R²'].mean(), metrics_df['MAE'].mean(), metrics_df['MSE'].mean()]
    })
    avg_metrics.to_excel(writer, sheet_name='Average Metrics', index=False)

print(f"十折交叉验证的预测结果和评估指标已保存到 {output_path_cv_results}")


with open('rf_best_model_daogao.pkl', 'wb') as f:
    pickle.dump(rf_best, f)  # 将最优模型保存到文件中
print('模型已保存到 rf_best_model_daogao.pkl')  # 打印保存成功信息




ValueError: could not convert string to float: 'B2'

In [ ]:
import numpy as np
import random
import copy
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


''' 种群初始化函数 '''
def initial(pop, dim, ub, lb):
    X = np.zeros([pop, dim])
    for i in range(pop):
        for j in range(dim):
            X[i, j] = random.random()*(ub[j] - lb[j]) + lb[j]
    
    return X,lb,ub
            
'''边界检查函数'''
def BorderCheck(X,ub,lb,pop,dim):
    for i in range(pop):
        for j in range(dim):
            if X[i,j]>ub[j]:
                X[i,j] = ub[j]
            elif X[i,j]<lb[j]:
                X[i,j] = lb[j]
    return X
    
    
'''计算适应度函数'''
def CaculateFitness(X,fun):
    pop = X.shape[0]
    fitness = np.zeros([pop, 1])
    for i in range(pop):
        fitness[i] = fun(X[i, :])
    return fitness

'''适应度排序'''
def SortFitness(Fit):
    fitness = np.sort(Fit, axis=0)
    index = np.argsort(Fit, axis=0)
    return fitness,index


'''根据适应度对位置进行排序'''
def SortPosition(X,index):
    Xnew = np.zeros(X.shape)
    for i in range(X.shape[0]):
        Xnew[i,:] = X[index[i],:]
    return Xnew

'''麻雀发现者更新'''
def PDUpdate(X,PDNumber,ST,Max_iter,dim):
    X_new  = copy.copy(X)
    R2 = random.random()
    for j in range(PDNumber):
        if R2<ST:
            X_new[j,:] = X[j,:]*np.exp(-j/(random.random()*Max_iter))
        else:
            X_new[j,:] = X[j,:] + np.random.randn()*np.ones([1,dim])
    return X_new
        
'''麻雀加入者更新'''            
def JDUpdate(X,PDNumber,pop,dim):
    X_new = copy.copy(X)
    for j in range(PDNumber+1,pop):
         if j>(pop - PDNumber)/2 + PDNumber:
             X_new[j,:]= np.random.randn()*np.exp((X[-1,:] - X[j,:])/j**2)
         else:
             #产生-1，1的随机数
             A = np.ones([dim,1])
             for a in range(dim):
                 if(random.random()>0.5):
                     A[a]=-1       
             AA = np.dot(A,np.linalg.inv(np.dot(A.T,A)))
             X_new[j,:]= X[0,:] + np.abs(X[j,:] - X[0,:])*AA.T
           
    return X_new                    
            
'''危险更新'''   
def SDUpdate(X,pop,SDNumber,fitness,BestF):
    X_new = copy.copy(X)
    Temp = range(pop)
    RandIndex = random.sample(Temp, pop)
    SDchooseIndex = RandIndex[0:SDNumber]
    for j in range(SDNumber):
        if fitness[SDchooseIndex[j]]>BestF:
            X_new[SDchooseIndex[j],:] = X[0,:] + np.random.randn()*np.abs(X[SDchooseIndex[j],:] - X[0,:])
        elif fitness[SDchooseIndex[j]] == BestF:
            K = 2*random.random() - 1
            X_new[SDchooseIndex[j],:] = X[SDchooseIndex[j],:] + K*(np.abs( X[SDchooseIndex[j],:] - X[-1,:])/(fitness[SDchooseIndex[j]] - fitness[-1] + 10E-8))
    return X_new
              
    

'''麻雀搜索算法'''
def SSA(pop,dim,lb,ub,Max_iter,fun):
    ST = 0.6 #预警值
    PD = 0.7 #发现者的比列，剩下的是加入者
    SD = 0.2 #意识到有危险麻雀的比重
    PDNumber = int(pop*PD) #发现者数量
    SDNumber = int(pop*SD) #意识到有危险麻雀数量
    X,lb,ub = initial(pop, dim, ub, lb) #初始化种群
    fitness = CaculateFitness(X,fun) #计算适应度值
    fitness,sortIndex = SortFitness(fitness) #对适应度值排序
    X = SortPosition(X,sortIndex) #种群排序
    GbestScore = copy.copy(fitness[0])
    GbestPositon = np.zeros([1,dim])
    GbestPositon[0,:] = copy.copy(X[0,:])
    Curve = np.zeros([Max_iter,1])
    for i in range(Max_iter):
        print("第"+str(i)+"次迭代")
        BestF = fitness[0]
        
        X = PDUpdate(X,PDNumber,ST,Max_iter,dim)#发现者更新
        
        X = JDUpdate(X,PDNumber,pop,dim) #加入者更新
        
        X = SDUpdate(X,pop,SDNumber,fitness,BestF) #危险更新
        
        X = BorderCheck(X,ub,lb,pop,dim) #边界检测
        
        fitness = CaculateFitness(X,fun) #计算适应度值
        fitness,sortIndex = SortFitness(fitness) #对适应度值排序
        X = SortPosition(X,sortIndex) #种群排序
        if(fitness[0]<=GbestScore): #更新全局最优
            GbestScore = copy.copy(fitness[0])
            GbestPositon[0,:] = copy.copy(X[0,:])
        Curve[i] = GbestScore
    
    return GbestScore,GbestPositon,Curve










In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import SSA as SSA
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
plt.close("all")



'''目标函数(适应度函数)'''
#定义适应函数，以测试集和训练集的绝对误差和为适应度值
def fun(X):
    #训练随机森林分类器
    N_estimators = int(X[0]) #随机森林个数 N_estimators 表示随机森林中树木的数量
    Max_features = int(X[1]) #最大特征数
    Model=RandomForestRegressor(n_estimators=N_estimators,max_features=Max_features, max_depth=None,min_samples_split=2, bootstrap=True,random_state=0)
    Model.fit(P_train,T_train)
    PredictTrain=Model.predict(P_train)
    PredictTest=Model.predict(P_test)
    MSETrain= np.sqrt(np.sum((PredictTrain - T_train)**2))/T_train.size#计算MSE
    MSETest=np.sqrt(np.sum((PredictTest - T_test)**2))/T_test.size#计算MSE
    output = MSETrain+MSETest
    return output
#读取数据,输入数据为2维的数据，输出数据为1维的数据

# 读取CSV文件
df = pd.read_csv('ChengDu H2024.csv')
df.columns = ['SECTION', 'LAYOUT', 'ACREAGE', 'DECORATION', 'FLOORS LEVEL', 'TFLOORS', 'AGE', 'TPRICE', 'UPRICE', 'LIFT']
csv_content = df.to_string(index=False)
# 选择4个特征作为X
Relation = ['SECTION', 'DECORATION','LAYOUT','FLOORS LEVEL','TFLOORS', 'AGE','LIFT','ACREAGE','UPRICE','TPRICE']
features = ['AGE','LIFT','ACREAGE','UPRICE','LAYOUT']
featurData = df[features].values
labelData = df['TPRICE'].values

# 绘制关联矩阵
import numpy as np
import seaborn as sns
# 将相关系数矩阵以热力图的形式可视化
cm = np.corrcoef(df[Relation].values.T)
#  cbar=True 表示显示颜色条，square=True 表示将热力图的宽高设置为相等，annot_kws={'size':} 表示热力图上的数值字体大小
hm = sns.heatmap(cm, cbar=True, square=True, fmt='.2f', annot=True, annot_kws={'size':11}, yticklabels=Relation, xticklabels=Relation)
plt.show()

# 划分训练集和测试集
split_point = int(len(featurData) * 0.8)
P_train = featurData[:split_point]
P_test = featurData[split_point:]
T_train = labelData[:split_point]
T_test = labelData[split_point:]

# 获取特征数
n_features = P_train.shape[1]

#设置麻雀参数
pop = 10 #种群数量
MaxIter = 20 #最大迭代次数 --算法运行的最多代数
dim = 2 #维度 --优化随机森林回归模型的两个参数n_estimators和max_features
lb = np.array([1,1]) #下边界
ub = np.array([100,n_features])#上边界
fobj = fun
GbestScore,GbestPositon,Curve = SSA.SSA(pop,dim,lb,ub,MaxIter,fobj) 
print('最优适应度值：',GbestScore)
print('N_estimators最优解：',int(GbestPositon[0,0]))
print('Max_features最优解：',int(GbestPositon[0,1]))
#利用最终优化的结果计算分类正确率等信息
#利用最优参数训练随机森林
N_estimators = int(GbestPositon[0,0]) #随机森林个数
Max_features = int(GbestPositon[0,1]) #最大特征数

''' 利用SSA改进之后的随机森林进行预测'''
ModelSSA=RandomForestRegressor(n_estimators=N_estimators,max_features=Max_features, max_depth=None,min_samples_split=2, bootstrap=True,random_state=42)
ModelSSA.fit(P_train,T_train)
PredictTrainSSA=ModelSSA.predict(P_train)
PredictTestSSA=ModelSSA.predict(P_test)
# 计算MSE
MSETrainSSA = np.sqrt(np.sum((PredictTrainSSA - T_train)**2)) / T_train.size
MSETestSSA = np.sqrt(np.sum((PredictTestSSA - T_test)**2)) / T_test.size
print("RF-SSA训练集MSE：" +str(MSETrainSSA) )
print("RF-SSA测试集MSE："+str(MSETestSSA) )
print("RF-SSA总MSE："+str(MSETestSSA+MSETrainSSA) )

''' 利用基础随机森林进行预测'''
#设置参数为：n_estimators=10,max_features=1
#创建随机森林

Model=RandomForestRegressor(n_estimators=10,max_features=1, random_state=42)
Model.fit(P_train,T_train)
PredictTrain=Model.predict(P_train)
PredictTest=Model.predict(P_test)
MSETrain= np.sqrt(np.sum((PredictTrain - T_train)**2))/T_train.size#计算MSE
MSETest=np.sqrt(np.sum((PredictTest - T_test)**2))/T_test.size#计算MSE
print("RF训练集MSE：" +str(MSETrain) )
print("RF测试集MSE："+str(MSETest) )
print("RF总MSE："+str(MSETest+MSETrain) )

#绘制适应度曲线
plt.figure(1)
plt.plot(Curve,'r-',linewidth=2)
plt.xlabel('Iteration',fontsize='medium')
plt.ylabel("Fitness",fontsize='medium')
plt.grid()
plt.title('SSA-RF',fontsize='large')
plt.show()
plt.savefig('../适应度曲线.png')

import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# 设置数据采样的间隔
sampling_interval = 42
# 对数据进行采样
sampled_indices = range(0, len(T_test), sampling_interval)
sampled_T_test = T_test[sampled_indices]
sampled_PredictTest = PredictTest[sampled_indices]
# 创建图形和子图
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(sampled_indices, sampled_T_test, label='True Values', linewidth=2)
ax.plot(sampled_indices, sampled_PredictTest, label='Predicted Values', linewidth=2)
ax.set_xlabel('Index', fontsize=14)
ax.set_ylabel('Value', fontsize=14)
ax.set_title(f'True Values vs Predicted Values (Model $R^2$ = {r2_score(T_test, PredictTest):.4f})', fontsize=16)
ax.legend()
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# 设置数据采样的间隔
sampling_interval = 42
# 对数据进行采样
sampled_indices = range(0, len(T_test), sampling_interval)
sampled_T_test = T_test[sampled_indices]
sampled_PredictTestSSA = PredictTestSSA[sampled_indices]
# 创建图形和子图
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(sampled_indices, sampled_T_test, label='True Values', linewidth=2)
ax.plot(sampled_indices, sampled_PredictTestSSA, label='Predicted Values', linewidth=2)
ax.set_xlabel('Index', fontsize=14)
ax.set_ylabel('Value', fontsize=14)
ax.set_title(f'True Values vs Predicted Values (Model $R^2$ = {r2_score(T_test, PredictTestSSA):.4f})', fontsize=16)
ax.legend()
plt.tight_layout()
plt.show()
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import r2_score

# 生成网格点
n_estimators_range = np.arange(10, 200, 10)
max_features_range = np.arange(1, P_train.shape[1], 1)
n_estimators_grid, max_features_grid = np.meshgrid(n_estimators_range, max_features_range)

# 计算不同参数组合下的R²
r2_scores = np.zeros_like(n_estimators_grid, dtype=float)
for i in range(n_estimators_grid.shape[0]):
    for j in range(n_estimators_grid.shape[1]):
        n_estimators = int(n_estimators_grid[i, j])
        max_features = int(max_features_grid[i, j])
        
        model = RandomForestRegressor(n_estimators=n_estimators, max_features=max_features,
                                      max_depth=None, min_samples_split=2, bootstrap=True, random_state=42)
        model.fit(P_train, T_train)
        
        predict_test = model.predict(P_test)
        
        r2 = r2_score(T_test, predict_test)
        r2_scores[i, j] = r2

# 创建一个新的图形和三维坐标轴
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 绘制三维曲面
ax.plot_surface(n_estimators_grid, max_features_grid, r2_scores, cmap='viridis', edgecolor='none')

# 设置坐标轴标签和标题
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_features')
ax.set_zlabel('R²')
ax.set_title('Parameter Space')

# 显示图形
plt.tight_layout()
plt.show()

In [1]:
import numpy as np
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.inspection import plot_partial_dependence

file_path = r'D:\Phyon\Project\bys\label_data_change1.xlsx'  # 读取表格
data = pd.read_excel(file_path,header=0)  # header为表头，自动去掉表头
data = data.values

N = 8
features = data[:, 1:N+1]  # 特征
labels_q = data[:, -3]     # 是否合格
labels_qd = data[:, -4]     # 偏差
labels_en = data[:, -2]     # 能耗


# 划分训练集，4:1
features_train, features_test,labels_q_train, labels_q_test,labels_qd_train, labels_qd_test,labels_en_train,\
labels_en_test = train_test_split(features, labels_q,labels_qd,labels_en,test_size=0.2, random_state=0)


'''
随机数种子是该组随机数的编号，在需要重复试验的时候，保证得到一组一样的随机数。比如你每次都填1，
其他参数一样的情况下你得到的随机数组是一样的。不填的话默认值为False，即每次切分的比例虽然相同，但是切分的结果不同
'''


# # 标准化
# from sklearn.preprocessing import StandardScaler
# ss_x,ss_y = StandardScaler(),StandardScaler()
# features_train = ss_x.fit_transform(features_train)
# features_test = ss_x.transform(features_test)
# labels_q_train = ss_y.fit_transform(labels_q_train.reshape([-1,1])).reshape(-1)
# labels_q_test = ss_y.transform(labels_q_test.reshape([-1,1])).reshape(-1)
# labels_qd_train = ss_y.fit_transform(labels_qd_train.reshape([-1,1])).reshape(-1)
# labels_qd_test = ss_y.transform(labels_qd_test.reshape([-1,1])).reshape(-1)
# labels_en_train = ss_y.fit_transform(labels_en_train.reshape([-1,1])).reshape(-1)
# labels_en_test = ss_y.transform(labels_en_test.reshape([-1,1])).reshape(-1)

from sklearn.metrics import mean_squared_error
def plot_learning_curve(reg, X_train, X_test, y_train, y_test):
    # 使用线性回归绘制学习曲线
    train_score = []
    test_score = []

    for i in range(1, len(X_train)):
        reg.fit(X_train[:i], y_train[:i])
        y_train_predict = reg.predict(X_train[:i])
        y_test_predict = reg.predict(X_test)
        train_score.append(mean_squared_error(y_train_predict, y_train[:i]))
        test_score.append(mean_squared_error(y_test_predict, y_test))

    plt.plot([i for i in range(1, len(X_train))], np.sqrt(train_score), label="train")
    plt.plot([i for i in range(1, len(X_train))], np.sqrt(test_score), label="test")

    plt.legend()
    plt.show()



# 随机森林
print('分类合格品')
print('开始训练随机森林 | ','时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
q_rf = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=0)
q_rf.fit(features_train, labels_q_train)
print('随机森林训练完毕 | ','训练集分数为',q_rf.score(features_train, labels_q_train),"验证集分数为", q_rf.score(features_test, labels_q_test))   # 决定系数R^2
print("--" * 100)

labels_q_predict = q_rf.predict(features_test)
labels_q_train_predict = q_rf.predict(features_train)

from sklearn.metrics import f1_score
f1 = f1_score (labels_q_test, labels_q_predict, labels=None, pos_label=1, average='binary', sample_weight=None)
f1_1 = f1_score (labels_q_train, labels_q_train_predict, labels=None, pos_label=1, average='binary', sample_weight=None)
feat_labels1 = ["余量","切换位置","周期","1速","2速","保压压力","最大压力","最大速度"]
feat_labels=pd.DataFrame(feat_labels1)
importances = q_rf.feature_importances_
importances_q = importances
indices = np.argsort(importances)[::-1] # 下标排序
print('特征排序：')
for f in range(8):
    print("%2d) %-*s %f" % \
          (f + 1, 30, feat_labels1[indices[f]], importances[indices[f]]))
print("--" * 100)



# 训练随机森林,模型输出为偏差
print('预测偏差')
from sklearn.ensemble import RandomForestRegressor
print('开始训练随机森林 | ','时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
qd_rf = RandomForestRegressor(n_estimators=51,max_depth=10,min_samples_split=3,criterion='absolute_error')
qd_rf.fit(features_train, labels_qd_train)
print('随机森林训练完毕 | ','训练集分数为',qd_rf.score(features_train, labels_qd_train),"验证集分数为", qd_rf.score(features_test, labels_qd_test))   # 决定系数R^2
print("--" * 100)

labels_qd_predict = qd_rf.predict(features_test)
r_qd = labels_qd_test - labels_qd_predict

labels_qd_train_predict = qd_rf.predict(features_train)

importances = qd_rf.feature_importances_
importances_qd = importances
indices = np.argsort(importances)[::-1] # 下标排序
print('特征排序：')
for f in range(8):
    print("%2d) %-*s %f" % \
          (f + 1, 30, feat_labels1[indices[f]], importances[indices[f]]))
print("--" * 100)


# 训练随机森林,模型输出为能耗
print('预测能耗')
print('开始训练随机森林 | ','时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
en_rf = RandomForestRegressor()
en_rf.fit(features_train, labels_en_train)
print('随机森林训练完毕 | ','训练集分数为',en_rf.score(features_train, labels_en_train),"验证集分数为", en_rf.score(features_test, labels_en_test))   # 决定系数R^2
print("--" * 100)

labels_en_predict = en_rf.predict(features_test)
r_en = labels_en_test - labels_en_predict
labels_en_train_predict = en_rf.predict(features_train)

importances = en_rf.feature_importances_
importances_en = importances
indices = np.argsort(importances)[::-1] # 下标排序
print('特征排序：')
for f in range(8):
    print("%2d) %-*s %f" % \
          (f + 1, 30, feat_labels1[indices[f]], importances[indices[f]]))
print("--" * 100)



features1 = np.array(features)
# 遗传算法优化
# 参数
DNA_SIZE = 12   # DNA长度与保留位数有关,长度越长精度越高,10的倍数
POP_ORI_SIZE = 500000  # 初始种群数量
CROSSOVER_RATE = 0.8    # 交叉概率
MUTATION_RATE = 0.01   # 变异概率
N_GENERATIONS = 100  # 迭代次数
X1_BOUND = [min(features1[:,0]), max(features1[:,0])]   # 范围
X2_BOUND = [min(features1[:,1]), max(features1[:,1])]
X3_BOUND = [min(features1[:,2]), max(features1[:,2])]
X4_BOUND = [min(features1[:,3]), max(features1[:,3])]
X5_BOUND = [min(features1[:,4]), max(features1[:,4])]
X6_BOUND = [min(features1[:,5]), max(features1[:,5])]   # 范围
X7_BOUND = [min(features1[:,6]), max(features1[:,6])]
X8_BOUND = [min(features1[:,7]), max(features1[:,7])]



# 求最小值适应度函数
def get_fitness(pop):
    x1,x2,x3,x4,x5,x6,x7,x8 = translateDNA(pop)
    a = [x1,x2,x3,x4,x5,x6,x7,x8]
    c = np.array(a)
    x = c.transpose()
    # pred1 = qd_rf.predict(x)
    pred11 = qd_rf.predict(x)
    pred22 = en_rf.predict(x)

    aa = 0
    bb = 1

    cc1 = min(pred11)
    dd1 = max(pred11)
    k1 = (bb-aa)/(dd1-cc1)
    pred1 = aa + k1 * (pred11 - cc1)

    cc2 = min(pred22)
    dd2 = max(pred22)
    k2 = (bb-aa)/(dd2-cc2)
    pred2 = aa + k2 * (pred22 - cc2)

    pred = 0.1*pred1 + 0.9*pred2
    return -(pred - np.max(pred)) + 1e-3


# 解码过程
def translateDNA(pop):  # pop表示种群矩阵，一行表示一个二进制编码表示的DNA，矩阵的行数为种群数目
    x1_pop = pop[:, ::8]   # 从第一个元素起，步长为8取元素
    x2_pop = pop[:, 1::8]  # 从第二个元素起，步长为8取元素
    x3_pop = pop[:, 2::8]  # 从第三个元素起，步长为8取元素
    x4_pop = pop[:, 3::8]  # 从第四个元素起，步长为8取元素
    x5_pop = pop[:, 4::8]  # 从第五个元素起，步长为8取元素
    x6_pop = pop[:, 5::8]   # 从第六个元素起，步长为8取元素
    x7_pop = pop[:, 6::8]  # 从第七个元素起，步长为8取元素
    x8_pop = pop[:, 7::8]  # 从第八个元素起，步长为8取元素

    x1 = x1_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X1_BOUND[1] - X1_BOUND[0]) + X1_BOUND[0]
    x2 = x2_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X2_BOUND[1] - X2_BOUND[0]) + X2_BOUND[0]
    x3 = x3_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X3_BOUND[1] - X3_BOUND[0]) + X3_BOUND[0]
    x4 = x4_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X4_BOUND[1] - X4_BOUND[0]) + X4_BOUND[0]
    x5 = x5_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X5_BOUND[1] - X5_BOUND[0]) + X5_BOUND[0]
    x6 = x6_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X6_BOUND[1] - X6_BOUND[0]) + X6_BOUND[0]
    x7 = x7_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X7_BOUND[1] - X7_BOUND[0]) + X7_BOUND[0]
    x8 = x8_pop.dot(2 ** np.arange(DNA_SIZE)[::-1]) / float(2 ** DNA_SIZE - 1) * (X8_BOUND[1] - X8_BOUND[0]) + X8_BOUND[0]

    return x1,x2,x3,x4,x5,x6,x7,x8


# 交叉过程（过程中产生变异）
def crossover_and_mutation(pop, CROSSOVER_RATE):
    new_pop = []
    for father in pop:  # 遍历种群中的每一个个体，将该个体作为父亲
        child = father  # 孩子先得到父亲的全部基因（这里我把一串二进制串的那些0，1称为基因）
        if np.random.rand() < CROSSOVER_RATE:  # 产生子代时不是必然发生交叉，而是以一定的概率发生交叉
            mother = pop[np.random.randint(POP_SIZE)]  # 再种群中选择另一个个体，并将该个体作为母亲
            cross_points = np.random.randint(low=0, high=DNA_SIZE * 2)  # 随机产生交叉的点
            child[cross_points:] = mother[cross_points:]  # 孩子得到位于交叉点后的母亲的基因
        mutation(child,MUTATION_RATE)  # 每个后代有一定的机率发生变异
        new_pop.append(child)

    return new_pop


# 变异过程
def mutation(child, MUTATION_RATE):
    if np.random.rand() < MUTATION_RATE:  # 以MUTATION_RATE的概率进行变异
        mutate_point = np.random.randint(0, DNA_SIZE * 2)  # 随机产生一个实数，代表要变异基因的位置
        child[mutate_point] = child[mutate_point] ^ 1  # 将变异点的二进制为反转


# 选择过程
def select(pop, fitness):  # nature selection wrt pop's fitness
    idx = np.random.choice(np.arange(POP_SIZE), size=POP_SIZE, replace=True,
                           p=(fitness) / (fitness.sum()))
    return pop[idx]


# 打印结果
def print_info(pop):
    fitness = get_fitness(pop)
    max_fitness_index = np.argmax(fitness)
    # print("max_fitness:", fitness[max_fitness_index])
    x1,x2,x3,x4,x5,x6,x7,x8 = translateDNA(pop)
    print('\n')
    print('遗传算法优化完毕 | ', '时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
    print("--" * 100)
    print("最优参数为:", (x1[max_fitness_index], x2[max_fitness_index],x3[max_fitness_index],x4[max_fitness_index],x5[max_fitness_index],x6[max_fitness_index], x7[max_fitness_index],x8[max_fitness_index]))
    print("最低偏差为:",qd_rf.predict([[x1[max_fitness_index], x2[max_fitness_index],x3[max_fitness_index],x4[max_fitness_index],x5[max_fitness_index],x6[max_fitness_index], x7[max_fitness_index],x8[max_fitness_index]]])[0])
    print("最低能耗为:",en_rf.predict([[x1[max_fitness_index], x2[max_fitness_index], x3[max_fitness_index], x4[max_fitness_index],x5[max_fitness_index],x6[max_fitness_index], x7[max_fitness_index],x8[max_fitness_index]]])[0])

en_obj = []
qd_obj = []

def obj(pop):
    fitness = get_fitness(pop)
    max_fitness_index = np.argmax(fitness)
    x1, x2, x3, x4, x5, x6, x7, x8 = translateDNA(pop)
    qd_obj.append(qd_rf.predict([[x1[max_fitness_index], x2[max_fitness_index], x3[max_fitness_index],
                                    x4[max_fitness_index], x5[max_fitness_index], x6[max_fitness_index],
                                    x7[max_fitness_index], x8[max_fitness_index]]])[0])
    en_obj.append(en_rf.predict([[x1[max_fitness_index], x2[max_fitness_index], x3[max_fitness_index],
                                    x4[max_fitness_index], x5[max_fitness_index], x6[max_fitness_index],
                                    x7[max_fitness_index], x8[max_fitness_index]]])[0])


xx,yy=np.shape(features)
features1 = np.zeros([xx,yy])  # 存放标准化到[0,4095]的数

def demoo(value):
    a=0
    b=4095   # 12位二进制数
    k = (b-a)/(max(value)-min(value))
    return [a+k*(x-min(value)) for x in value]


for j in range(yy):
    features1[:,j] = demoo(features[:,j])

features2 = np.round(features1)  # 四舍五入

x11 = features2[:,0]
x22 = features2[:,1]
x33 = features2[:,2]
x44 = features2[:,3]
x55 = features2[:,4]
x66 = features2[:,5]
x77 = features2[:,6]
x88 = features2[:,7]

x1 = [bin(int(x11[i]))[2:] for i in range(xx)]
x2 = [bin(int(x22[i]))[2:] for i in range(xx)]
x3 = [bin(int(x33[i]))[2:] for i in range(xx)]
x4 = [bin(int(x44[i]))[2:] for i in range(xx)]
x5 = [bin(int(x55[i]))[2:] for i in range(xx)]
x6 = [bin(int(x66[i]))[2:] for i in range(xx)]
x7 = [bin(int(x77[i]))[2:] for i in range(xx)]
x8 = [bin(int(x88[i]))[2:] for i in range(xx)]

xxx = np.array([x1,x2,x3,x4,x5,x6,x7,x8])
xxxx = xxx.transpose()

for j in range(yy):
    for i in range(xx):
        l = len(xxxx[i,j])
        pr = 12-l
        pr0 = pr*'0'
        xxxx[i,j]= '{}{}'.format(pr0,xxxx[i,j])


x_initial = np.zeros([xx,12*N])

jj = 0
for i in range(0,12*N,12):
    for j in range(xx):
        x_initial[j, i] = int(xxxx[j,jj][0])
        x_initial[j, i + 1] = int(xxxx[j, jj][1])
        x_initial[j, i + 2] = int(xxxx[j, jj][2])
        x_initial[j, i + 3] = int(xxxx[j, jj][3])
        x_initial[j, i + 4] = int(xxxx[j, jj][4])
        x_initial[j, i + 5] = int(xxxx[j, jj][5])
        x_initial[j, i + 6] = int(xxxx[j, jj][6])
        x_initial[j, i + 7] = int(xxxx[j, jj][7])
        x_initial[j, i + 8] = int(xxxx[j, jj][8])
        x_initial[j, i + 9] = int(xxxx[j, jj][9])
        x_initial[j, i + 10] = int(xxxx[j, jj][10])
        x_initial[j, i + 11] = int(xxxx[j, jj][11])
    jj = jj+1

x_initial1 = np.zeros([xx,12*N])

x_initial1[:, 0::8] = x_initial[:,0:12]
x_initial1[:, 1::8] = x_initial[:, 12:24]
x_initial1[:, 2::8] = x_initial[:, 24:36]
x_initial1[:, 3::8] = x_initial[:, 36:48]
x_initial1[:, 4::8] = x_initial[:, 48:60]
x_initial1[:, 5::8] = x_initial[:, 60:72]
x_initial1[:, 6::8] = x_initial[:, 72:84]
x_initial1[:, 7::8] = x_initial[:, 84:96]

x_initial2 = x_initial1.astype(int)

# 优化
print('开始进行遗传算法优化 | ','时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())),'\n')
# pop = np.random.randint(2, size=(POP_ORI_SIZE, DNA_SIZE * N))   # 随机生成基因型
pop = x_initial2
POP_SIZE = len(x_initial1)
for i in range(N_GENERATIONS):  # 迭代N代

    if POP_SIZE != POP_ORI_SIZE:
        xx,yy = pop.shape
        pop_add = np.random.randint(2, size=(POP_ORI_SIZE-POP_SIZE, DNA_SIZE * N))
        pop = np.vstack((pop, pop_add))
        POP_SIZE = POP_ORI_SIZE
    x1, x2, x3, x4, x5, x6, x7, x8 = translateDNA(pop)
    fitness = get_fitness(pop)
    max_fitness_index1 = np.argmax(fitness)
    max_fitness1=fitness[max_fitness_index1]
    pop = select(pop, fitness)  # 选择生成新的种群
    pop = np.array(crossover_and_mutation(pop, CROSSOVER_RATE))  # 进行选择

    x1, x2, x3, x4, x5, x6, x7, x8 = translateDNA(pop)
    POP_SIZE1 = POP_SIZE
    # 根据质量是否合格剔除个体
    a = [x1,x2,x3,x4,x5,x6,x7,x8]  #
    de1 = [index for index in range(POP_ORI_SIZE) if (x4[index]-x5[index])<5]
    c = np.array(a)
    x = c.transpose()
    pp = q_rf.predict(x)  # 预测质量是否合格
    # print(pp)
    de2 = [index for (index, value) in enumerate(pp) if value == 0]  # 记录不合格的个体的索引值
    de = list(set(de1+de2))
    pop = np.delete(pop, de, axis=0)  # 剔除个体
    POP_SIZE = POP_SIZE1 - len(de)
    obj(pop)

    # if (i+1)%10==0:
    # print('已完成'+str(int((i+1)*100/N_GENERATIONS))+'% | ', '时间：', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())))
    # print('已迭代'+str(i+1)+'代 | ','共剔除'+str(POP_ORI_SIZE-POP_SIZE)+'个样本，'+'当前种群数量为：', str(POP_SIZE), '\n')

    print('已迭代'+str(i+1)+'代|'+'共'+str(N_GENERATIONS)+'代'+'|当前最优的适应度函数值为'+str(max_fitness1))

print_info(pop)
# plt.plot([i for i in range(1, len(en_obj)+1)], en_obj, label="能耗")
# plt.legend()
# plt.show()
#
# plt.plot([i for i in range(1, len(qd_obj)+1)], qd_obj, label="偏差")
# plt.legend()
# plt.show()
#

ImportError: cannot import name 'plot_partial_dependence' from 'sklearn.inspection' (C:\Users\马明杰\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\inspection\__init__.py)